# Feature Extraction 08: Sentence-Embedding Text Signal

Trains a supervised "sentence analyzer" directly on StockTwits message *text*, using each
message's realized forward abnormal return as the training label, then aggregates the
model's predicted score to stock-day level as new x-variables to test against next-day
returns.

**Relationship to other notebooks**
- `features_06_full_text_exploration.ipynb` validated the `messages/` + `msg_info/` join and
  prototyped cashtag/mention extraction (Features 49-51 groundwork). This notebook is a
  different track: instead of hand-built text features, it lets a model learn directly
  from raw message text.
- `features_07_user_influence_accuracy.ipynb` scores *users* by track record with a
  walk-forward, no-look-ahead design. This notebook applies the same discipline to *text*:
  the "skill" being learned lives in a sentence-embedding regression instead of a per-user
  hit rate.

**Method**
1. Encode each message's cleaned body with a pretrained sentence-transformer
   (`all-MiniLM-L6-v2`, 384-dim).
2. Fit an online linear model (`SGDRegressor`, updated via `partial_fit`) that predicts
   `ar_capm_1` (next-trading-day CAPM abnormal return) from the embedding.
3. Walk forward year by year: a year's messages are *scored* using only the model fit on
   strictly earlier years, then the model is updated with that year's realized outcomes
   before moving on to the next year -- no look-ahead.
4. Aggregate the message-level predicted scores to (`symbol`, `date`) level: mean, std
   (dispersion), and count -- the x-variables saved by this notebook.

**Why an online (`partial_fit`) model instead of the expanding-window retrain used in
`features_07`:** the embedding matrix for the full 15-year corpus (100M+ messages x 384
float32 dims) would be on the order of a couple hundred GB if held in memory across years.
An online linear model lets us process one year of embeddings at a time and discard them
immediately after the `partial_fit` call, at the cost of a simpler (linear, not fully
re-fit) learner.

**Scale warning -- read before running Section 8:** encoding is CPU-only in this
environment (no GPU detected: `torch` reports `cpu` build). Section 7 benchmarks
throughput on real sample data and extrapolates a wall-clock estimate for the full
corpus -- expect this to be on the order of days, not hours, of continuous compute.
Sections 3 and 8 are both checkpointed (per source file / per year) so the notebook can
be safely interrupted and resumed across multiple sessions.

## 1. Setup and Configuration

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from tqdm.notebook import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import SGDRegressor

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR      = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR      = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
MESSAGES_DIR  = DATA_DIR / "messages"                  # message_id, message_body (205 files, NOT year-chunked)
RETURNS_DIR   = DATA_DIR / "merged_with_crsp_mlcrowd"  # message-level, per-year, has message_id + ar_capm_*
OUTPUT_FOLDER = DATA_DIR / "features_mlcrowd"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE   = OUTPUT_FOLDER / "features_08_text_embedding_signal.pkl"

# Intermediate/checkpoint storage -- large, kept separate from the small pickle outputs
WORK_DIR         = DATA_DIR / "features_08_workdir"
JOINED_DIR       = WORK_DIR / "joined_by_year"       # per-year csv: message_id, message_body, metadata, targets
EMBED_CACHE_DIR  = WORK_DIR / "embeddings_by_year"    # optional per-year cached embeddings (.npy)
MODEL_STATE_DIR  = WORK_DIR / "model_state"           # walk-forward SGDRegressor checkpoint after each year
YEAR_FEATURE_DIR = WORK_DIR / "features_by_year"      # per-year aggregated stock-day features (resumable Section 8)
for d in (JOINED_DIR, EMBED_CACHE_DIR, MODEL_STATE_DIR, YEAR_FEATURE_DIR):
    d.mkdir(parents=True, exist_ok=True)
JOIN_PASS_LOG = WORK_DIR / "join_pass_processed_files.txt"

# =============================================================================
# PARAMETERS
# =============================================================================
TARGET_COL        = "ar_capm_1"    # next-trading-day CAPM abnormal return (the "next day return" label)
EMBED_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM         = 384
ENCODE_BATCH_SIZE = 256
KEEP_EMBEDDINGS   = False          # True keeps per-year .npy embeddings on disk for reuse/debugging (large!)
SGD_PARAMS = dict(
    loss="squared_error", penalty="l2", alpha=1e-4,
    learning_rate="invscaling", eta0=0.01, power_t=0.25,
    random_state=42,
)

print(f"Messages dir  : {MESSAGES_DIR}")
print(f"Returns dir   : {RETURNS_DIR}")
print(f"Output file   : {OUTPUT_FILE}")
print(f"Work dir      : {WORK_DIR}")
print(f"Target col    : {TARGET_COL}")
print(f"Embed model   : {EMBED_MODEL_NAME} ({EMBED_DIM}-dim)")

import torch
print(f"torch device  : {'cuda' if torch.cuda.is_available() else 'cpu'} (build: {torch.__version__})")

## 2. Build the Message Universe

`merged_with_crsp_mlcrowd/` already restricts to Bullish/Bearish-labeled, CRSP-matched
messages and is chunked by year, but it has no message text. Load just the join key and
labels (not the full 43-column file) across all years, so we know exactly which
`message_id`s are worth encoding and which year bucket each belongs to.

In [ ]:
RETURN_FILES = sorted(RETURNS_DIR.glob("stocktwits_crsp_*.csv"))
print(f"Found {len(RETURN_FILES)} year files in {RETURNS_DIR.name}")

NEEDED_COLS = ["message_id", "user_id", "date", "symbol", "sentiment", "ar_capm_1", "ar_capm_21"]

universe_frames = []
for f in tqdm(RETURN_FILES, desc="Loading message universe"):
    year = int(f.stem.split("_")[-1])
    df_y = pd.read_csv(f, usecols=NEEDED_COLS)
    df_y["year"] = year
    universe_frames.append(df_y)

universe = pd.concat(universe_frames, ignore_index=True)
universe["date"] = pd.to_datetime(universe["date"])
universe["message_id"] = pd.to_numeric(universe["message_id"], errors="coerce")
universe = universe.dropna(subset=["message_id"])
universe["message_id"] = universe["message_id"].astype("int64")
universe = universe.drop_duplicates(subset="message_id")

print(f"Universe size: {len(universe):,} messages across {universe['year'].nunique()} years")
print(universe.groupby("year").size())

## 3. Pass 1 -- Join Message Text into the Universe (checkpointed)

`messages/` is NOT year-chunked (per `features_06`'s exploration -- 205 files whose
`message_id` ranges span many calendar years). So this streams every `messages/msg_*.csv`
file once, inner-joins each chunk's `message_id` against `universe`, and appends matches to
a per-year CSV under `JOINED_DIR`. Finished source files are recorded in `JOIN_PASS_LOG`,
so re-running this cell after a full completion is a no-op, and interrupting mid-run only
wastes at most one file's work.

A small fraction of message bodies contain stray/unterminated quote characters that make
pandas' default C parser raise a hard `ParserError` ("buffer overflow") partway through a
file rather than just corrupting one row (as `features_06` saw on a non-chunked read).
`engine="python"` with `on_bad_lines="skip"` tolerates these and drops the offending rows
instead of crashing the whole join pass.

In [ ]:
def _load_processed_files():
    if JOIN_PASS_LOG.exists():
        return set(JOIN_PASS_LOG.read_text().splitlines())
    return set()

def _mark_processed(fname):
    with open(JOIN_PASS_LOG, "a") as fh:
        fh.write(fname + "\n")

universe_indexed = universe.set_index("message_id")
message_files = sorted(MESSAGES_DIR.glob("msg_*.csv"))
processed = _load_processed_files()
todo_files = [f for f in message_files if f.name not in processed]

print(f"messages/ files: {len(message_files)} total, {len(processed)} already joined, "
      f"{len(todo_files)} remaining")

CHUNK_SIZE = 200_000
_open_writers = {}  # year -> (file handle, still-need-header bool)

def _get_writer(year):
    if year not in _open_writers:
        path = JOINED_DIR / f"joined_{year}.csv"
        needs_header = not path.exists()
        _open_writers[year] = [open(path, "a", newline="", encoding="utf-8"), needs_header]
    return _open_writers[year]

for f in tqdm(todo_files, desc="Joining messages/ files"):
    # engine="python" + on_bad_lines="skip": a small fraction of message bodies contain
    # stray/unterminated quote characters that make the (much faster) C engine raise a
    # hard "buffer overflow" ParserError partway through the file (observed on msg_000.csv
    # around row ~600k). The python engine tolerates these and just drops the bad rows.
    reader = pd.read_csv(
        f, usecols=["message_id", "message_body"], chunksize=CHUNK_SIZE,
        engine="python", on_bad_lines="skip",
    )
    for chunk in reader:
        chunk["message_id"] = pd.to_numeric(chunk["message_id"], errors="coerce")
        chunk = chunk.dropna(subset=["message_id"])
        chunk["message_id"] = chunk["message_id"].astype("int64")

        joined = chunk.join(universe_indexed, on="message_id", how="inner")
        if joined.empty:
            continue

        for year, grp in joined.groupby("year"):
            writer = _get_writer(int(year))
            grp.drop(columns="year").to_csv(writer[0], header=writer[1], index=False)
            writer[1] = False

    _mark_processed(f.name)

for fh, _ in _open_writers.values():
    fh.close()

print("Join pass complete (or already complete from a prior run).")
print("Joined year files:", sorted(p.name for p in JOINED_DIR.glob("joined_*.csv")))

## 4. Core Functions: Text Cleaning, Embedding, Aggregation

In [ ]:
import html
import re

WHITESPACE_RE = re.compile(r"\s+")

def clean_text(body):
    # Unescape HTML entities and collapse whitespace; keep cashtags/mentions as
    # context for the embedding model rather than stripping them.
    return WHITESPACE_RE.sub(" ", html.unescape(str(body))).strip()

def load_year(year):
    path = JOINED_DIR / f"joined_{year}.csv"
    df = pd.read_csv(path)
    df["message_id"] = df["message_id"].astype("int64")
    df["date"] = pd.to_datetime(df["date"])
    df["message_body"] = df["message_body"].apply(clean_text)
    return df

def embed_texts(model, texts, batch_size=ENCODE_BATCH_SIZE):
    return model.encode(
        list(texts), batch_size=batch_size, show_progress_bar=False,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")

def aggregate_to_stock_day(df, scores, score_col="text_signal"):
    # Aggregate message-level predicted scores to (symbol, date) x-variables.
    out = df[["symbol", "date"]].copy()
    out[score_col] = scores
    agg = out.groupby(["symbol", "date"])[score_col].agg(
        **{
            f"{score_col}_mean": "mean",
            f"{score_col}_std": "std",
            f"{score_col}_n": "count",
        }
    ).reset_index()
    return agg

## 5. Walk-Forward Driver

For year `Y`, in order:
1. Load `Y`'s joined text + targets.
2. Encode all of `Y`'s message bodies.
3. **Predict first** using the model as it exists *before* touching `Y` (i.e. fit only on
   years `< Y`) -- these predictions are the genuinely out-of-sample x-variable. The first
   year processed has no prior fit, so its scores are left as `NaN` (cold start, same
   convention as `features_07`'s no-skill prior).
4. Aggregate predicted scores to stock-day level.
5. `partial_fit` the model on `Y`'s (embedding, `TARGET_COL`) pairs, then discard the
   embeddings (unless `KEEP_EMBEDDINGS`) before moving to `Y+1`.

In [ ]:
def process_year(year, model, sgd, is_first_fit):
    df = load_year(year)
    df = df.dropna(subset=[TARGET_COL])

    t0 = time.time()
    embeddings = embed_texts(model, df["message_body"].tolist())
    encode_secs = time.time() - t0

    if is_first_fit:
        scores = np.full(len(df), np.nan, dtype="float32")
    else:
        scores = sgd.predict(embeddings)

    agg = aggregate_to_stock_day(df, scores)
    agg["year"] = year

    sgd.partial_fit(embeddings, df[TARGET_COL].to_numpy())

    if KEEP_EMBEDDINGS:
        np.save(EMBED_CACHE_DIR / f"embed_{year}.npy", embeddings)

    stats = {
        "year": year, "n_messages": len(df), "n_stock_days": len(agg),
        "encode_secs": encode_secs, "msgs_per_sec": len(df) / max(encode_secs, 1e-9),
    }
    return agg, stats

## 6. Validate on a Single Year (2010 -- the smallest year)

Runs the real pipeline end-to-end (embedding model, cold-start scoring, `partial_fit`,
aggregation) on the smallest available year before committing to the full run.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
sgd_validate = SGDRegressor(**SGD_PARAMS)

years_available = sorted(int(p.stem.split("_")[-1]) for p in JOINED_DIR.glob("joined_*.csv"))
print(f"Years available after the join pass: {years_available}")

first_year = years_available[0]
agg_first, stats_first = process_year(first_year, embed_model, sgd_validate, is_first_fit=True)

print(f"\nYear {first_year} (cold start -- scores should be all-NaN):")
print(stats_first)
display(agg_first.head(10))
assert agg_first["text_signal_mean"].isna().all(), "Cold-start year should have no scores yet"

second_year = years_available[1]
agg_second, stats_second = process_year(second_year, embed_model, sgd_validate, is_first_fit=False)

print(f"\nYear {second_year} (first year with real out-of-sample scores):")
print(stats_second)
display(agg_second.head(10))
assert agg_second["text_signal_mean"].notna().any(), "Second year should produce real scores"
print("\nValidation passed: cold-start + first real prediction + partial_fit update all work.")

## 7. Runtime Benchmark & Full-Corpus Estimate

**Read this before running Section 8.** Encoding is CPU-only here. This benchmarks
observed throughput from Section 6 and extrapolates a wall-clock estimate for the full
`universe` built in Section 2, so the actual cost is known before committing to it.

In [ ]:
total_messages = len(universe)
observed_rate = stats_second["msgs_per_sec"]  # from the real validation run, not a toy benchmark

est_seconds = total_messages / observed_rate
est_hours = est_seconds / 3600

print(f"Observed encoding rate ({second_year}, CPU): {observed_rate:,.1f} messages/sec")
print(f"Total messages in universe: {total_messages:,}")
print(f"Estimated full-corpus encoding time: {est_hours:,.1f} hours (~{est_hours/24:,.1f} days)")
print()
print("This is a lower bound: it excludes I/O for the join pass (Section 3, ~52 GB of raw")
print("text, already a one-time cost) and assumes throughput stays constant across years")
print("(later years have far more messages per file, so cache/memory pressure may differ).")
print()
print("Section 8 is checkpointed per year via YEAR_FEATURE_DIR + MODEL_STATE_DIR: it is")
print("safe to run it for a while, stop, and resume later across multiple sessions.")

## 8. Process All Years (Walk-Forward, Checkpointed)

Resumable at year granularity: on each run, finds the latest year with a saved model
checkpoint and resumes the walk-forward from the next year. A year only counts as "done"
once *both* its feature parquet and its model checkpoint are written -- if a prior run
was interrupted between those two writes, that year is simply redone (at most one year's
worth of re-encoding), rather than silently skipping its `partial_fit` update, which would
otherwise leave the model checkpoint claiming to reflect data it never actually saw.

In [ ]:
def _year_feature_path(year):
    return YEAR_FEATURE_DIR / f"features_{year}.parquet"

def _model_checkpoint_path(year):
    return MODEL_STATE_DIR / f"sgd_after_{year}.joblib"

years_available = sorted(int(p.stem.split("_")[-1]) for p in JOINED_DIR.glob("joined_*.csv"))
done_years = sorted(
    int(p.stem.split("_")[-1]) for p in MODEL_STATE_DIR.glob("sgd_after_*.joblib")
)

if done_years:
    resume_from_year = done_years[-1]
    sgd = joblib.load(_model_checkpoint_path(resume_from_year))
    print(f"Resuming: loaded model checkpoint fit through {resume_from_year}")
else:
    resume_from_year = None
    sgd = SGDRegressor(**SGD_PARAMS)
    print("Starting fresh: no prior model checkpoint found")

remaining_years = [y for y in years_available if resume_from_year is None or y > resume_from_year]
print(f"Years remaining to process: {remaining_years}")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
run_stats = []

for year in tqdm(remaining_years, desc="Walk-forward years"):
    feat_path = _year_feature_path(year)
    model_path = _model_checkpoint_path(year)
    is_first_fit = resume_from_year is None and year == remaining_years[0]

    if feat_path.exists() and model_path.exists():
        # Both artifacts for this year survived a prior run -- truly done, safe to skip.
        print(f"  {year}: already complete (features + model checkpoint present), skipping")
        continue

    # Otherwise (co)compute from scratch for this year -- if only one of the two files
    # exists from an interrupted prior run, both get overwritten here so the model
    # checkpoint always reflects a partial_fit that actually happened.
    agg, stats = process_year(year, embed_model, sgd, is_first_fit=is_first_fit)
    agg.to_parquet(feat_path, index=False)
    joblib.dump(sgd, model_path)
    run_stats.append(stats)
    print(f"  {year}: {stats['n_messages']:,} msgs -> {stats['n_stock_days']:,} stock-days "
          f"({stats['msgs_per_sec']:.1f} msg/s)")

print("\nAll years processed (or resumed to completion).")
if run_stats:
    display(pd.DataFrame(run_stats))

## 9. Combine All Years & Final Inspection

In [ ]:
year_files = sorted(YEAR_FEATURE_DIR.glob("features_*.parquet"))
features_all = pd.concat([pd.read_parquet(f) for f in year_files], ignore_index=True)
features_all = features_all.drop(columns="year").sort_values(["date", "symbol"]).reset_index(drop=True)

print(f"Shape: {features_all.shape}")
print(f"Columns: {list(features_all.columns)}")
print(f"Date range: {features_all['date'].min().date()} to {features_all['date'].max().date()}")
print(f"Unique symbols: {features_all['symbol'].nunique()}")
print(f"\nNull counts:")
print(features_all.isnull().sum())
print(f"\nSummary statistics:")
display(features_all.describe().round(4))

## 10. Save to Pickle

In [ ]:
print(f"Saving to: {OUTPUT_FILE}")
features_all.to_pickle(OUTPUT_FILE)

verify = pd.read_pickle(OUTPUT_FILE)
assert verify.shape == features_all.shape
file_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"Saved and verified. File size: {file_mb:.1f} MB")

## Summary

**Rationale**

Every other feature in this project treats a message's self-reported `sentiment` tag
(Bullish/Bearish) as the unit of signal. That discards the actual text -- two "Bullish"
messages can carry very different information (a one-word "$AAPL bullish" vs. a detailed
thesis). This notebook lets a model read the raw text and learn, directly from realized
forward returns, which patterns in message content are associated with subsequent price
moves -- independent of, and potentially complementary to, the hand-labeled sentiment tag.

**Methodology**

1. Each message is embedded with a small pretrained sentence-transformer
   (`all-MiniLM-L6-v2`, 384-dim, L2-normalized output).
2. An online linear model (`SGDRegressor`) is trained via `partial_fit` to predict
   `ar_capm_1` (next-trading-day CAPM abnormal return) from the embedding.
3. Walk-forward, year by year: each year's messages are scored using only the model state
   from strictly earlier years (no look-ahead), then the model is updated with that year's
   realized outcomes.
4. Message-level scores are aggregated to (`symbol`, `date`): `text_signal_mean` (the
   crowd's average text-implied expected return), `text_signal_std` (dispersion --
   analogous to `skill_dispersion` in `features_07`), and `text_signal_n` (message count
   backing the aggregate).

**Known limitations**

- A single message's forward return is the *stock's* return, not something the message
  itself caused -- most of the variance in any one message's label is unrelated to its
  text. The model is expected to explain very little individual-message variance; the
  aggregated stock-day mean is the intended signal, analogous to averaging noisy analyst
  forecasts.
- The online (`partial_fit`) design trades model capacity (linear only, one pass per year)
  for the ability to process 100M+ messages without holding the full embedding matrix in
  memory. A future iteration could revisit this with dimensionality-reduced embeddings and
  a periodically-refit (rather than purely online) model if capacity turns out to be the
  binding constraint.
- Encoding is CPU-only in this environment; see Section 7's runtime estimate before
  running Section 8 on the full corpus.

**Next steps**

- Merge `features_08_text_embedding_signal.pkl` into `merged_master.pkl` alongside the
  other feature files (`02 - prepare training dataset/merge_all_feature_files.ipynb`).
- Compare `text_signal_mean` against `net_sentiment` (features_01) and
  `skill_wtd_net_sent_*` (features_07) for correlation and incremental R^2 in the
  downstream prediction notebooks (`03a`-`03e`).
- Per `features_06`'s roadmap, this is one of several open-ended full-text directions
  (topic modeling, sentiment-lexicon validation); those remain unexplored.